# Emerging Technologies Problem Set


In [32]:
import numpy as np
from itertools import product

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit import transpile

## Problem 1: Generating Random Boolean Functions


Deutsch's algorithm is trying to solve the problem of determining whether a function is balanced or constant given input bit `x = 0`, and input function `f(x) = {0,1}`. If a problem is constant it will always return either true or false for all 2^n inputs. If its balanced it will return 'True' exactly half of the time, and 'False' the other half.

These special 'black box functions' are called oracles meaning you cannot inspect its internals.

Classically the solution that uses a deterministic algorithm where n is the number of bits, it will take `2^(n-1) + 1` evaluations in the worst case scenario. Meaning just over half of the inputs will always need to be evaluated to prove a function is constant[1]. 

[1] https://en.wikipedia.org/wiki/Deutsch%E2%80%93Jozsa_algorithm#Classical_solution


In [33]:
def random_constant_balanced():
    '''
    Returns a randomly chosen oracle functions that takes in 4 booleans as its inputs

    The oracle can be either :
    - Constant: returns True or False for all 16 outputs
    - Balanced: returns True for exactly half of the inputs

    Each function type has equal probability of being chosen
    '''

    # Generating constant or balanced input function
    is_constant = np.random.choice([True,False])

    if is_constant:
        # Generating inputs for constant function
        constant_value = np.random.choice([True, False])
        def f(b1, b2, b3, b4):
            return constant_value
    else:
        # Balanced Oracle
        # Select exactly 8 of 16 input indices to return True
        all_indices = np.arange(16)
        true_indices = np.random.choice(all_indices, size=8, replace=False)
        true_set = set(true_indices)
        
        def f(b1, b2, b3, b4):
            # Convert 4 booleans to index 0-15
            index = (int(b1) << 3) | (int(b2) << 2) | (int(b3) << 1) | int(b4)

            # Checking if the converted binary inputs in the function are in the true set
            return index in true_set
    
    return f

Converting the 4 inputs in the functions are done by mapping each boolean to a bit `True -> 1, False -> 0` then converting the binary number to decimal.

`eg. (T,F,T,F) -> 0b1010 -> 10`

### Testing the function


In [34]:
all_inputs = list(product([True, False], repeat=4))  

counts = {"constant": 0, "balanced": 0}

# Testing 1000 functions
for _ in range(1000):
    f = random_constant_balanced()
    results = [f(*inp) for inp in all_inputs]
    true_count = results.count(True)
    
    # Updating count for balanced and constant functions
    if true_count in (0, 16):
        counts["constant"] += 1
    else:
        counts["balanced"] += 1

print(counts)

{'constant': 535, 'balanced': 465}


For 1000 functions, we can expect that rougly 500 of each function can be expected as the output because each oracle is generated with equal probability. eg `{'constant': 484, 'balanced': 516}`

## Problem 2: Classical Testing for Function Type


In [35]:
def determine_constant_balanced(f):
    all_inputs = list(product([True, False], repeat=4))
    first_result = f(*all_inputs[0])
    
    for i, inp in enumerate(all_inputs[1:], start=1):
        result = f(*inp)
        if result != first_result:
            return "balanced"
        if i + 1 == 9:
            return "constant"
    
    return "constant"

In [36]:
correct = 0
total = 1000

for _ in range(total):
    f = random_constant_balanced()
    results = [f(*inp) for inp in all_inputs]
    true_count = results.count(True)
    ground_truth = "constant" if true_count in (0, 16) else "balanced"
    
    if determine_constant_balanced(f) == ground_truth:
        correct += 1

print(f"{correct}/{total} correct")

1000/1000 correct


## Problem 3: Quantum Oracles


In [37]:
def oracle_constant_0():
    qc = QuantumCircuit(2, name="f=0")
    return qc

def oracle_constant_1():
    qc = QuantumCircuit(2, name="f=1")
    qc.x(1)
    return qc

def oracle_identity():
    qc = QuantumCircuit(2, name="f=x")
    qc.cx(0, 1)
    return qc

def oracle_not():
    qc = QuantumCircuit(2, name="f=NOT x")
    qc.cx(0, 1)
    qc.x(1)
    return qc

In [38]:
simulator = AerSimulator()

def evaluate_oracle(oracle_fn, name):
    print(f"\n{name}")
    print(oracle_fn().draw())
    for x in [0, 1]:
        qc = QuantumCircuit(2, 1)
        if x == 1:
            qc.x(0)
        qc.compose(oracle_fn(), inplace=True)
        qc.measure(1, 0)
        
        counts = simulator.run(transpile(qc, simulator), shots=1).result().get_counts()
        print(f"  f({x}) = {list(counts.keys())[0]}")

In [39]:

evaluate_oracle(oracle_constant_0, "Constant-0: f(x) = 0")
evaluate_oracle(oracle_constant_1, "Constant-1: f(x) = 1")
evaluate_oracle(oracle_identity,   "Identity:   f(x) = x")
evaluate_oracle(oracle_not,        "NOT:        f(x) = NOT x")


Constant-0: f(x) = 0
     
q_0: 
     
q_1: 
     
  f(0) = 0
  f(1) = 0

Constant-1: f(x) = 1
          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘
  f(0) = 1
  f(1) = 1

Identity:   f(x) = x
          
q_0: ──■──
     ┌─┴─┐
q_1: ┤ X ├
     └───┘
  f(0) = 0
  f(1) = 1

NOT:        f(x) = NOT x
               
q_0: ──■───────
     ┌─┴─┐┌───┐
q_1: ┤ X ├┤ X ├
     └───┘└───┘
  f(0) = 1
  f(1) = 0


In [40]:
def deutsch_algorithm(oracle_fn, name):
    qc = QuantumCircuit(2, 1)
    
    qc.x(1)
    
    qc.h(0)
    qc.h(1)
    
    #Apply oracle
    qc.compose(oracle_fn(), inplace=True)
    
    qc.h(0)
    
    #Measure input qubit
    qc.measure(0, 0)
    
    counts = simulator.run(transpile(qc, simulator), shots=1).result().get_counts()
    result = list(counts.keys())[0]
    classification = "constant" if result == "0" else "balanced"
    print(f"{name:30s} ->  measured |{result}⟩  -> {classification}")
    return qc

print("\nDeutsch Algorithm Results:")
for fn, name in [(oracle_constant_0, "f(x) = 0"),
                 (oracle_constant_1, "f(x) = 1"),
                 (oracle_identity,   "f(x) = x"),
                 (oracle_not,        "f(x) = NOT x")]:
    deutsch_algorithm(fn, name)


Deutsch Algorithm Results:
f(x) = 0                       ->  measured |0⟩  -> constant
f(x) = 1                       ->  measured |0⟩  -> constant
f(x) = x                       ->  measured |1⟩  -> balanced
f(x) = NOT x                   ->  measured |1⟩  -> balanced


## Problem 4: Deutsch's Algorithm


The quantum circuit has 2 qubits and 1 classical bit.
q1 is set to `|1⟩`, after performing a hadamard operation it is set to `|-⟩`, this enables phase kickback later.

Hadamard is applied to both qubits. For q0 this creates an equal superposition of `|0⟩` and `|1⟩`(this represents both 0 and 1 as inputs simultaneously)

The oracle is applied to both qubits. q0's inputs are processed simultaneously. The `|-⟩` state on q1 cause phase kickback. The oracle encodes f(x) into phase of q(0). If f is constant both terms get the same phases. If f is balanced the terms pick up opposite phases.

Hadamard is applied to q0 again to convert the encoded phase difference to a measurable probability amplitude.

if f was constant the phases were the same. constructive interference `q0 -> |0⟩`.
if f was balanaced the phases were the opposite. destructive interference `q0 -> |1⟩`.


In [41]:
# Deutsch's algorithm circuit — works for any single-bit oracle
# Circuit structure: 2 qubits, 1 classical bit


def deutsch_algorithm(oracle_fn, name):
    qc = QuantumCircuit(2, 1)

    qc.x(1)

    qc.h(0)
    qc.h(1)

    qc.compose(oracle_fn(), inplace=True)

    qc.h(0)

    qc.measure(0, 0)

    counts = simulator.run(transpile(qc, simulator), shots=1).result().get_counts()
    result = list(counts.keys())[0]
    classification = "constant" if result == "0" else "balanced"
    print(f"{name:30s} → measured |{result}⟩ → {classification}")
    return qc

In [42]:
print("Deutsch Algorithm Results:")

oracles = [
    (oracle_constant_0, "f(x) = 0"),
    (oracle_constant_1, "f(x) = 1"),
    (oracle_identity,   "f(x) = x"),
    (oracle_not,        "f(x) = NOT x")
]

for fn, name in oracles:
    qc = deutsch_algorithm(fn, name)

#Drawing circuit for identity oracle
print("\nExample circuit (f(x) = x):")
qc_draw = QuantumCircuit(2, 1)
qc_draw.x(1)
qc_draw.h(0)
qc_draw.h(1)
qc_draw.compose(oracle_identity(), inplace=True)
qc_draw.h(0)
qc_draw.measure(0, 0)
print(qc_draw.draw())

Deutsch Algorithm Results:
f(x) = 0                       → measured |0⟩ → constant
f(x) = 1                       → measured |0⟩ → constant
f(x) = x                       → measured |1⟩ → balanced
f(x) = NOT x                   → measured |1⟩ → balanced

Example circuit (f(x) = x):
     ┌───┐          ┌───┐┌─┐
q_0: ┤ H ├───────■──┤ H ├┤M├
     ├───┤┌───┐┌─┴─┐└───┘└╥┘
q_1: ┤ X ├┤ H ├┤ X ├──────╫─
     └───┘└───┘└───┘      ║ 
c: 1/═════════════════════╩═
                          0 


## Problem 5: Scaling to the Deutsch-Josza Algorithm
